# 01 Candidate records and the overlap assessment (gate G1)

Design log.
- Candidate records: 67 items from the scoping note and the second search of 19 Nov 2026 (surveys, SoKs, evaluation studies, one benchmark, one item that is not a survey), written to `schema/candidates_seed.yaml` and built into `reports/candidates.csv` with preprint and published versions linked by title or DOI.
- arXiv metadata (title, first and latest version dates, journal reference, DOI) for the 56 arXiv items through the arXiv API, two batched requests, cached in `reports/arxiv_meta.json`.
- PDFs: the 56 arXiv PDFs at their latest version and 3 open-access journal PDFs are downloaded into `data/surveys/` (not committed), one request every three seconds; sha256 and page count go into `reports/surveys.csv`. Paywalled or unlinked items are expected as hand-placed files `data/surveys/<id>.pdf` and are reported present or missing.
- Overlap forms: one YAML form per closest item under `reports/overlap/`, prefilled from `schema/overlap_prefill.yaml` (what was read on 19 Nov 2026, with the page or basis it came from) only when the form does not exist yet, so filled forms survive reruns. A form counts as confirmed only when `confirmed_from_pdf` is true, `pdf_sha256` is filled and every partial or yes answer has a page.
- Decision (`reports/decision_01.txt`, `reports/overlap_01.md`): UNRESOLVED while any form is unconfirmed or invalid; NARROW if a confirmed form answers yes to any of the three questions; SUPPORTED otherwise. The rule is in `src/overlap.py` with tests.
- Outside Colab, `SOURCEMAP_OFFLINE=1` skips the network and the notebook runs on the seed alone; this is how it was dry-run.
- CPU; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd())

## Stage 1. Modules, tests and the committed seed data

In [ ]:
%%writefile src/corpus.py
"""Candidate records, arXiv metadata, PDF downloads and hashes for the survey corpus."""
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import time
import urllib.error
import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path

ARXIV_API = "https://export.arxiv.org/api/query"
ATOM = {"a": "http://www.w3.org/2005/Atom", "arxiv": "http://arxiv.org/schemas/atom"}
USER_AGENT = "sourcemap-research corpus builder"
OFFLINE = os.environ.get("SOURCEMAP_OFFLINE") == "1"
REQUEST_GAP_SECONDS = 3.0

CANDIDATE_FIELDS = ["id", "kind", "title", "first_author", "arxiv", "doi", "venue", "first_date", "access", "pdf_url",
                    "origin", "closest", "arxiv_title", "arxiv_published", "arxiv_updated", "arxiv_latest_version",
                    "arxiv_journal_ref", "arxiv_doi", "linked_to"]
SURVEY_FIELDS = CANDIDATE_FIELDS + ["pdf_path", "pdf_status", "sha256", "pages"]


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def page_count(path: Path):
    try:
        from pypdf import PdfReader
        return len(PdfReader(str(path)).pages)
    except Exception:
        return None


def normalise_title(title: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", (title or "").lower()).strip()


def read_csv(path: Path) -> list[dict]:
    if not Path(path).exists():
        return []
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write_csv(path: Path, rows: list[dict], fieldnames: list[str]) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for row in rows:
            w.writerow({k: row.get(k, "") for k in fieldnames})


def link_versions(rows: list[dict]) -> list[dict]:
    """Mark records that share a normalised title or a DOI as versions of one study (linked_to = the first id)."""
    seen_title: dict[str, str] = {}
    seen_doi: dict[str, str] = {}
    for row in rows:
        key_t = normalise_title(row.get("title", ""))
        key_d = (row.get("doi") or "").lower().strip()
        first = seen_title.get(key_t) or (seen_doi.get(key_d) if key_d else None)
        root = first if first and first != row["id"] else row["id"]
        row["linked_to"] = "" if root == row["id"] else root
        seen_title.setdefault(key_t, root)
        if key_d:
            seen_doi.setdefault(key_d, root)
    return rows


def _get(url: str, timeout: int = 60) -> bytes:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()


def parse_arxiv_feed(xml_bytes: bytes) -> dict:
    """Entries of an arXiv API Atom feed keyed by the versionless arXiv id."""
    out = {}
    root = ET.fromstring(xml_bytes)
    for entry in root.findall("a:entry", ATOM):
        full_id = (entry.findtext("a:id", default="", namespaces=ATOM) or "").rsplit("/", 1)[-1]
        m = re.match(r"(.+?)v(\d+)$", full_id)
        base, version = (m.group(1), int(m.group(2))) if m else (full_id, None)
        if not base:
            continue
        out[base] = {
            "title": " ".join((entry.findtext("a:title", default="", namespaces=ATOM) or "").split()),
            "published": (entry.findtext("a:published", default="", namespaces=ATOM) or "")[:10],
            "updated": (entry.findtext("a:updated", default="", namespaces=ATOM) or "")[:10],
            "latest_version": version,
            "authors": [a.findtext("a:name", default="", namespaces=ATOM) for a in entry.findall("a:author", ATOM)],
            "journal_ref": entry.findtext("arxiv:journal_ref", default="", namespaces=ATOM) or "",
            "doi": entry.findtext("arxiv:doi", default="", namespaces=ATOM) or "",
            "comment": entry.findtext("arxiv:comment", default="", namespaces=ATOM) or "",
        }
    return out


def fetch_arxiv_metadata(ids: list[str], cache_path: Path, batch: int = 40) -> dict:
    """Title, dates, latest version, journal reference and DOI for each arXiv id, cached as JSON."""
    cache_path = Path(cache_path)
    cache = json.loads(cache_path.read_text()) if cache_path.exists() else {}
    missing = [i for i in ids if i and i not in cache]
    if OFFLINE or not missing:
        return cache
    for start in range(0, len(missing), batch):
        chunk = missing[start:start + batch]
        url = f"{ARXIV_API}?id_list={','.join(chunk)}&max_results={len(chunk)}"
        try:
            cache.update(parse_arxiv_feed(_get(url)))
        except (urllib.error.URLError, ET.ParseError, OSError) as exc:
            print("arXiv API request failed:", exc)
            break
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        cache_path.write_text(json.dumps(cache, indent=1, ensure_ascii=False))
        time.sleep(REQUEST_GAP_SECONDS)
    return cache


def download_pdf(url: str, dest: Path) -> str:
    """Download to dest; returns 'downloaded', 'present', 'offline' or 'failed: <reason>'."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 1000:
        return "present"
    if OFFLINE:
        return "offline"
    try:
        data = _get(url, timeout=120)
    except Exception as exc:  # network and HTTP errors alike are reported, not raised
        return f"failed: {exc}"
    if not data.startswith(b"%PDF"):
        return "failed: response is not a PDF"
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(data)
    time.sleep(REQUEST_GAP_SECONDS)
    return "downloaded"


def pdf_url_for(row: dict) -> str:
    if row.get("arxiv"):
        version = row.get("arxiv_latest_version") or ""
        return f"https://arxiv.org/pdf/{row['arxiv']}" + (f"v{version}" if version else "")
    return row.get("pdf_url") or ""

In [ ]:
%%writefile src/overlap.py
"""Overlap assessment forms for the closest reviews, and the decision rule of gate G1."""
from __future__ import annotations

from pathlib import Path

import yaml

ANSWER_VALUES = {"yes", "partial", "no", "unknown"}
TEXT_FIELDS = ["scope", "corpus_cutoff", "organising_dimensions", "treatment_user_task", "treatment_document_handling",
               "treatment_comparability", "evidence_granularity", "what_this_review_adds"]
ANSWER_FIELDS = ["a_source_axis", "b_three_jobs_together", "c_equal_layout_rule"]
QUESTIONS = {
    "a_source_axis": "organises defences by where the model is given the instruction's source",
    "b_three_jobs_together": "reports security, the user's task and document handling together",
    "c_equal_layout_rule": "states or applies a rule that numbers are comparable only in the same prompt layout",
}


def blank_form(record_id: str, title: str, kind: str) -> dict:
    form = {"id": record_id, "title": title, "kind": kind}
    for f in TEXT_FIELDS:
        form[f] = ""
    for f in ANSWER_FIELDS:
        form[f] = "unknown"
    form["pages"] = {"a": "", "b": "", "c": ""}
    form["read_basis"] = ""
    form["confirmed_from_pdf"] = False
    form["pdf_sha256"] = ""
    return form


def write_form_if_missing(path: Path, form: dict) -> bool:
    path = Path(path)
    if path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(form, sort_keys=False, allow_unicode=True, width=110))
    return True


def load_forms(folder: Path) -> list[dict]:
    forms = []
    for path in sorted(Path(folder).glob("*.yaml")):
        form = yaml.safe_load(path.read_text()) or {}
        form["_path"] = str(path)
        forms.append(form)
    return forms


def validate(form: dict) -> list[str]:
    problems = []
    for f in ["id", "title", "kind", "read_basis"]:
        if not form.get(f):
            problems.append(f"{f} missing")
    for f in TEXT_FIELDS:
        if not str(form.get(f, "")).strip():
            problems.append(f"{f} empty")
    for f in ANSWER_FIELDS:
        if form.get(f) not in ANSWER_VALUES:
            problems.append(f"{f} must be one of {sorted(ANSWER_VALUES)}")
    if form.get("confirmed_from_pdf"):
        if not form.get("pdf_sha256"):
            problems.append("confirmed_from_pdf is true but pdf_sha256 is empty")
        for key, f in zip("abc", ANSWER_FIELDS):
            if form.get(f) in {"yes", "partial"} and not str((form.get("pages") or {}).get(key, "")).strip():
                problems.append(f"{f} is {form.get(f)} but pages.{key} is empty")
        if any(form.get(f) == "unknown" for f in ANSWER_FIELDS):
            problems.append("confirmed_from_pdf is true but an answer is still unknown")
    return problems


def decide(forms: list[dict]) -> tuple[str, list[str]]:
    """UNRESOLVED if any form is unconfirmed or invalid; NARROW if any confirmed form answers yes; else SUPPORTED."""
    reasons = []
    unconfirmed = [f["id"] for f in forms if not f.get("confirmed_from_pdf")]
    invalid = {f["id"]: validate(f) for f in forms}
    invalid = {k: v for k, v in invalid.items() if v}
    if invalid:
        reasons += [f"{k}: {'; '.join(v)}" for k, v in invalid.items()]
    if unconfirmed:
        reasons.append("not yet confirmed from the PDF: " + ", ".join(unconfirmed))
    if invalid or unconfirmed:
        return "UNRESOLVED", reasons
    narrow = []
    for f in forms:
        for field in ANSWER_FIELDS:
            if f.get(field) == "yes":
                narrow.append(f"{f['id']} answers yes to {field} ({QUESTIONS[field]}); that claim must be narrowed or dropped")
    if narrow:
        return "NARROW", narrow
    return "SUPPORTED", ["no confirmed form answers yes to any of the three questions"]


def render_markdown(forms: list[dict], outcome: str, reasons: list[str], when: str) -> str:
    lines = [f"# Overlap assessment (gate G1), {when}", "", f"Outcome: **{outcome}**", ""]
    lines += [f"- {r}" for r in reasons] + [""]
    lines += ["| id | kind | a: source axis | b: three jobs | c: layout rule | confirmed | read basis |",
              "|---|---|---|---|---|---|---|"]
    for f in forms:
        lines.append(f"| {f.get('id')} | {f.get('kind')} | {f.get('a_source_axis')} | {f.get('b_three_jobs_together')} | "
                     f"{f.get('c_equal_layout_rule')} | {'yes' if f.get('confirmed_from_pdf') else 'no'} | {f.get('read_basis')} |")
    lines.append("")
    for f in forms:
        lines += [f"## {f.get('id')}: {f.get('title')}", ""]
        for field in TEXT_FIELDS:
            lines.append(f"- **{field.replace('_', ' ')}**: {f.get(field, '')}")
        pages = f.get("pages") or {}
        lines.append(f"- **pages**: a {pages.get('a', '')}; b {pages.get('b', '')}; c {pages.get('c', '')}")
        lines.append("")
    return "\n".join(lines)

In [ ]:
%%writefile tests/test_overlap.py

"""The gate G1 decision rule."""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
import overlap  # noqa: E402


def confirmed(record_id, a="no", b="no", c="no"):
    form = overlap.blank_form(record_id, "t", "survey")
    for f in overlap.TEXT_FIELDS:
        form[f] = "x"
    form.update(a_source_axis=a, b_three_jobs_together=b, c_equal_layout_rule=c, read_basis="full PDF",
                confirmed_from_pdf=True, pdf_sha256="abc")
    form["pages"] = {"a": "3", "b": "4", "c": "5"}
    return form


def test_unresolved_when_a_form_is_unconfirmed():
    forms = [confirmed("x"), overlap.blank_form("y", "t", "survey")]
    outcome, reasons = overlap.decide(forms)
    assert outcome == "UNRESOLVED"
    assert any("not yet confirmed" in r for r in reasons)


def test_narrow_when_a_confirmed_form_answers_yes():
    outcome, reasons = overlap.decide([confirmed("x"), confirmed("y", b="yes")])
    assert outcome == "NARROW"
    assert "b_three_jobs_together" in reasons[0]


def test_supported_when_all_confirmed_and_no_yes():
    outcome, _ = overlap.decide([confirmed("x", a="partial"), confirmed("y", c="partial")])
    assert outcome == "SUPPORTED"


def test_confirmed_form_needs_pages_for_partial_answers():
    form = confirmed("x", a="partial")
    form["pages"]["a"] = ""
    assert any("pages.a" in p for p in overlap.validate(form))

In [ ]:
%%writefile tests/test_corpus.py

"""Title normalisation, version linking and arXiv feed parsing."""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
import corpus  # noqa: E402

FEED = b"""<?xml version="1.0" encoding="UTF-8"?>
<feed xmlns="http://www.w3.org/2005/Atom" xmlns:arxiv="http://arxiv.org/schemas/atom">
  <entry>
    <id>http://arxiv.org/abs/2510.15476v4</id>
    <updated>2026-10-02T12:00:00Z</updated>
    <published>2025-10-17T09:00:00Z</published>
    <title>SoK: Systematizing LLM Prompt Security:
  Taxonomies, Datasets, and Unified Evaluation</title>
    <author><name>First Author</name></author>
    <author><name>Second Author</name></author>
    <arxiv:comment>v4</arxiv:comment>
  </entry>
  <entry>
    <id>http://arxiv.org/abs/2505.00976v2</id>
    <updated>2025-12-01T00:00:00Z</updated>
    <published>2025-05-02T00:00:00Z</published>
    <title>Attack and defense techniques in LLMs</title>
    <arxiv:journal_ref>Neural Networks 2025</arxiv:journal_ref>
    <arxiv:doi>10.1016/j.neunet.2025.108388</arxiv:doi>
    <author><name>Only Author</name></author>
  </entry>
</feed>
"""


def test_normalise_title_ignores_case_and_punctuation():
    assert corpus.normalise_title("SoK: Prompt Hacking of LLMs!") == corpus.normalise_title("sok prompt hacking of llms")


def test_link_versions_links_same_title_and_same_doi_to_one_root():
    rows = [
        {"id": "a", "title": "A Survey on X", "doi": ""},
        {"id": "b", "title": "A survey on X.", "doi": "10.1/x"},
        {"id": "c", "title": "Different", "doi": "10.1/X"},
        {"id": "d", "title": "Unrelated", "doi": ""},
    ]
    out = corpus.link_versions(rows)
    assert out[0]["linked_to"] == ""
    assert out[1]["linked_to"] == "a"
    assert out[2]["linked_to"] == "a"
    assert out[3]["linked_to"] == ""


def test_parse_arxiv_feed_reads_version_dates_and_journal_fields():
    meta = corpus.parse_arxiv_feed(FEED)
    assert set(meta) == {"2510.15476", "2505.00976"}
    hong = meta["2510.15476"]
    assert hong["latest_version"] == 4
    assert hong["published"] == "2025-10-17" and hong["updated"] == "2026-10-02"
    assert hong["title"].startswith("SoK: Systematizing LLM Prompt Security: Taxonomies")
    assert hong["authors"] == ["First Author", "Second Author"]
    liao = meta["2505.00976"]
    assert liao["journal_ref"] == "Neural Networks 2025" and liao["doi"] == "10.1016/j.neunet.2025.108388"


def test_pdf_url_uses_latest_version_when_known():
    assert corpus.pdf_url_for({"arxiv": "2510.15476", "arxiv_latest_version": 4}) == "https://arxiv.org/pdf/2510.15476v4"
    assert corpus.pdf_url_for({"arxiv": "2510.15476", "arxiv_latest_version": ""}) == "https://arxiv.org/pdf/2510.15476"
    assert corpus.pdf_url_for({"arxiv": "", "pdf_url": "https://x/y.pdf"}) == "https://x/y.pdf"
    assert corpus.pdf_url_for({"arxiv": "", "pdf_url": ""}) == ""

In [ ]:
%%writefile schema/candidates_seed.yaml
- id: rossi_2402.00898
  kind: survey
  title: An Early Categorization of Prompt Injection Attacks on Large Language Models
  first_author: Rossi
  arxiv: '2402.00898'
  doi: ''
  venue: arXiv
  first_date: '2024-01-31'
  access: arxiv
  origin: scoping
  closest: 0
- id: liu_hu_2406.00240
  kind: survey
  title: 'Exploring Vulnerabilities and Protections in Large Language Models: A Survey'
  first_author: Liu
  arxiv: '2406.00240'
  doi: ''
  venue: arXiv
  first_date: '2024-06-01'
  access: arxiv
  origin: scoping
  closest: 0
- id: rababah_2410.13901
  kind: sok
  title: 'SoK: Prompt Hacking of Large Language Models'
  first_author: Rababah
  arxiv: '2410.13901'
  doi: ''
  venue: arXiv
  first_date: '2024-10-16'
  access: arxiv
  origin: scoping
  closest: 0
- id: beurer_kellner_2506.08837
  kind: pattern
  title: Design Patterns for Securing LLM Agents against Prompt Injections
  first_author: Beurer-Kellner
  arxiv: '2506.08837'
  doi: ''
  venue: arXiv
  first_date: '2025-06-10'
  access: arxiv
  origin: scoping
  closest: 0
- id: mchugh_2507.13169
  kind: survey
  title: 'Prompt Injection 2.0: Hybrid AI Threats'
  first_author: McHugh
  arxiv: '2507.13169'
  doi: ''
  venue: arXiv
  first_date: '2025-07-17'
  access: arxiv
  origin: scoping
  closest: 0
- id: hong_2510.15476
  kind: sok
  title: 'SoK: Systematizing LLM Prompt Security: Taxonomies, Datasets, and Unified Evaluation of Attacks and Defenses'
  first_author: Hong
  arxiv: '2510.15476'
  doi: ''
  venue: arXiv
  first_date: '2025-10-17'
  access: arxiv
  origin: scoping
  closest: 1
- id: ji_2511.15203
  kind: sok
  title: Taxonomy, Evaluation and Exploitation of IPI-Centric LLM Agent Defense Frameworks
  first_author: Ji
  arxiv: '2511.15203'
  doi: ''
  venue: arXiv
  first_date: '2025-11-19'
  access: arxiv
  origin: scoping
  closest: 1
- id: geng_cmc_2026
  kind: survey
  title: 'Prompt Injection Attacks on Large Language Models: A Survey of Attack Methods, Root Causes, and Defense Strategies'
  first_author: Geng
  arxiv: ''
  doi: 10.32604/cmc.2025.074081
  venue: Computers, Materials & Continua 87(1)
  first_date: '2026-02-10'
  access: open
  pdf_url: https://www.techscience.com/cmc/v87n1/66084/pdf
  origin: scoping
  closest: 1
- id: maloyan_2601.17548
  kind: sok
  title: 'Prompt Injection Attacks on Agentic Coding Assistants: A Systematic Analysis of Vulnerabilities in Skills, Tools,
    and Protocol Ecosystems'
  first_author: Maloyan
  arxiv: '2601.17548'
  doi: ''
  venue: arXiv
  first_date: '2026-01-24'
  access: arxiv
  origin: scoping
  closest: 0
- id: wang_2602.10453
  kind: sok
  title: 'The Landscape of Prompt Injection Threats in LLM Agents: From Taxonomy to Analysis'
  first_author: Wang
  arxiv: '2602.10453'
  doi: ''
  venue: arXiv
  first_date: '2026-02-11'
  access: arxiv
  origin: scoping
  closest: 1
- id: deep_2604.23887
  kind: evaluation
  title: Evaluation of Prompt Injection Defenses in LLMs
  first_author: Deep
  arxiv: '2604.23887'
  doi: ''
  venue: arXiv
  first_date: '2026-04-26'
  access: arxiv
  origin: scoping
  closest: 0
- id: damianakis_2610.24999
  kind: sok
  title: 'SoK: PIAtlas: Decomposing the Unbounded Prompt Injection Attack Space'
  first_author: Damianakis
  arxiv: '2610.24999'
  doi: ''
  venue: arXiv
  first_date: '2026-10-18'
  access: arxiv
  origin: scoping
  closest: 0
- id: deng_2406.02630
  kind: survey
  title: 'AI Agents Under Threat: A Survey of Key Security Challenges and Future Pathways'
  first_author: Deng
  arxiv: '2406.02630'
  doi: 10.1145/3716628
  venue: ACM Computing Surveys
  first_date: '2024-06-04'
  access: arxiv
  origin: scoping
  closest: 0
- id: he_y_2406.08689
  kind: survey
  title: Security of AI Agents
  first_author: He
  arxiv: '2406.08689'
  doi: ''
  venue: arXiv
  first_date: '2024-06-12'
  access: arxiv
  origin: scoping
  closest: 0
- id: he_f_2407.19354
  kind: survey
  title: 'The Emerged Security and Privacy of LLM Agent: A Survey with Case Studies'
  first_author: He
  arxiv: '2407.19354'
  doi: 10.1145/3773080
  venue: ACM (journal not verified)
  first_date: '2024-07-28'
  access: arxiv
  origin: scoping
  closest: 0
- id: gan_2411.09523
  kind: survey
  title: 'Navigating the Risks: A Survey of Security, Privacy, and Ethics Threats in LLM-Based Agents'
  first_author: Gan
  arxiv: '2411.09523'
  doi: ''
  venue: arXiv
  first_date: '2024-11-14'
  access: arxiv
  origin: scoping
  closest: 0
- id: yu_2503.09648
  kind: survey
  title: 'A Survey on Trustworthy LLM Agents: Threats and Countermeasures'
  first_author: Yu
  arxiv: '2503.09648'
  doi: ''
  venue: KDD 2025 (per citing papers; not on the arXiv page)
  first_date: '2025-03-12'
  access: arxiv
  origin: scoping
  closest: 0
- id: wang_k_2504.15585
  kind: survey
  title: 'A Comprehensive Survey in LLM(-Agent) Full Stack Safety: Data, Training and Deployment'
  first_author: Wang
  arxiv: '2504.15585'
  doi: ''
  venue: arXiv
  first_date: '2025-04-22'
  access: arxiv
  origin: scoping
  closest: 0
- id: raza_2506.04133
  kind: survey
  title: 'TRiSM for Agentic AI: A Review of Trust, Risk, and Security Management in LLM-based Agentic Multi-Agent Systems'
  first_author: Raza
  arxiv: '2506.04133'
  doi: ''
  venue: arXiv (a 2026 journal version exists; journal not verified)
  first_date: '2025-06-04'
  access: arxiv
  origin: scoping
  closest: 0
- id: ferrag_2506.23260
  kind: survey
  title: 'From Prompt Injections to Protocol Exploits: Threats in LLM-Powered AI Agents Workflows'
  first_author: Ferrag
  arxiv: '2506.23260'
  doi: 10.1016/j.icte.2025.12.001
  venue: ICT Express
  first_date: '2025-06-29'
  access: arxiv
  origin: scoping
  closest: 0
- id: shahriar_2510.06445
  kind: survey
  title: 'A Survey on Agentic Security: Applications, Threats and Defenses'
  first_author: Shahriar
  arxiv: '2510.06445'
  doi: ''
  venue: TMLR 2026
  first_date: '2025-10-07'
  access: arxiv
  origin: scoping
  closest: 0
- id: chhabra_2510.23883
  kind: survey
  title: 'Agentic AI Security: Threats, Defenses, Evaluation, and Open Challenges'
  first_author: Chhabra
  arxiv: '2510.23883'
  doi: 10.1109/ACCESS.2026.3675554
  venue: IEEE Access 2026
  first_date: '2025-10-27'
  access: arxiv
  origin: scoping
  closest: 0
- id: kim_2603.11088
  kind: survey
  title: 'The Attack and Defense Landscape of Agentic AI: A Comprehensive Survey'
  first_author: Kim
  arxiv: '2603.11088'
  doi: ''
  venue: arXiv (cited elsewhere as USENIX Security 2026; not verified)
  first_date: '2026-03-11'
  access: arxiv
  origin: scoping
  closest: 0
- id: dehghantanha_2603.22928
  kind: sok
  title: 'SoK: The Attack Surface of Agentic AI: Tools and Autonomy'
  first_author: Dehghantanha
  arxiv: '2603.22928'
  doi: ''
  venue: arXiv
  first_date: '2026-03-24'
  access: arxiv
  origin: scoping
  closest: 0
- id: sun_preprints_202602.1655
  kind: survey
  title: A Survey on the Unique Security of Autonomous and Collaborative LLM Agents
  first_author: Sun
  arxiv: ''
  doi: 10.20944/preprints202602.1655.v2
  venue: Preprints.org (not peer reviewed)
  first_date: '2026-03-10'
  access: open
  pdf_url: ''
  origin: scoping
  closest: 0
- id: chu_2604.23338
  kind: survey
  title: 'A Systematic Survey of Security Threats and Defenses in LLM-Based AI Agents: A Layered Attack Surface Framework'
  first_author: Chu
  arxiv: '2604.23338'
  doi: ''
  venue: arXiv
  first_date: '2026-04-25'
  access: arxiv
  origin: scoping
  closest: 0
- id: xu_2606.28450
  kind: survey
  title: LLM agents security duality
  first_author: Xu
  arxiv: '2606.28450'
  doi: 10.1007/s10462-026-11563-0
  venue: Artificial Intelligence Review
  first_date: '2026-06-26'
  access: arxiv
  origin: scoping
  closest: 0
- id: hossain_2608.10530
  kind: survey
  title: On Understanding, Identifying, and Mitigating Vulnerabilities in Agentic Large Language Models
  first_author: Hossain
  arxiv: '2608.10530'
  doi: ''
  venue: arXiv (states under review at ACM Computing Surveys)
  first_date: '2026-08-11'
  access: arxiv
  origin: scoping
  closest: 0
- id: sammartino_kbs_2026
  kind: survey
  title: 'Vulnerabilities in autonomous execution: A survey of security threats and defenses in LLM-driven agentic and multi-agent
    systems'
  first_author: Sammartino
  arxiv: ''
  doi: 10.1016/j.knosys.2026.117001
  venue: Knowledge-Based Systems 352:117001
  first_date: '2026-09-25'
  access: paywalled
  pdf_url: ''
  origin: scoping
  closest: 1
- id: truong_le_2611.10843
  kind: sok
  title: 'Agents of Chaos: A SoK on Malicious Attacks on Agentic AI Systems'
  first_author: Truong-Le
  arxiv: '2611.10843'
  doi: ''
  venue: AIDC 2026 workshop at ACSAC 2026
  first_date: '2026-11-07'
  access: arxiv
  origin: scoping
  closest: 0
- id: yumusak_applsci_2026
  kind: survey
  title: 'Below the Waterline: A Survey of Agentic AI Security, Shadow AI Governance, and LLM Supply-Chain Risk in the Enterprise'
  first_author: Yumusak
  arxiv: ''
  doi: ''
  venue: Applied Sciences 16(20):10412 (seen in a citation list only)
  first_date: '2026-10-21'
  access: unknown
  pdf_url: ''
  origin: scoping
  closest: 0
- id: trust_auth_2512.06914
  kind: sok
  title: 'SoK: Trust-Authorization Mismatch in LLM Agent Interactions'
  first_author: ''
  arxiv: '2512.06914'
  doi: ''
  venue: arXiv
  first_date: 2025-12
  access: arxiv
  origin: scoping
  closest: 0
- id: mcp_2512.08290
  kind: sok
  title: SoK on the Model Context Protocol
  first_author: ''
  arxiv: '2512.08290'
  doi: ''
  venue: arXiv
  first_date: 2025-12
  access: arxiv
  origin: scoping
  closest: 0
- id: commerce_2604.15367
  kind: sok
  title: 'SoK: Security of Autonomous LLM Agents in Agentic Commerce'
  first_author: ''
  arxiv: '2604.15367'
  doi: ''
  venue: arXiv
  first_date: 2026-04
  access: arxiv
  origin: scoping
  closest: 0
- id: multiagent_2609.00595
  kind: sok
  title: 'SoK: When Safe Agents Fail Together'
  first_author: ''
  arxiv: '2609.00595'
  doi: ''
  venue: arXiv
  first_date: 2026-09
  access: arxiv
  origin: scoping
  closest: 0
- id: yao_2312.02003
  kind: survey
  title: 'A Survey on Large Language Model (LLM) Security and Privacy: The Good, the Bad, and the Ugly'
  first_author: Yao
  arxiv: '2312.02003'
  doi: 10.1016/j.hcc.2024.100211
  venue: High-Confidence Computing 2024
  first_date: '2023-12-04'
  access: arxiv
  origin: scoping
  closest: 0
- id: das_2402.00888
  kind: survey
  title: 'Security and Privacy Challenges of Large Language Models: A Survey'
  first_author: Das
  arxiv: '2402.00888'
  doi: ''
  venue: arXiv (journal version reported as ACM Computing Surveys 2025; not verified)
  first_date: '2024-01-30'
  access: arxiv
  origin: scoping
  closest: 0
- id: chowdhury_2403.04786
  kind: survey
  title: 'Breaking Down the Defenses: A Comparative Survey of Attacks on Large Language Models'
  first_author: Chowdhury
  arxiv: '2403.04786'
  doi: ''
  venue: arXiv
  first_date: '2024-03-03'
  access: arxiv
  origin: scoping
  closest: 0
- id: cui_2409.03274
  kind: survey
  title: Recent Advances in Attack and Defense Approaches of Large Language Models
  first_author: Cui
  arxiv: '2409.03274'
  doi: 10.1007/s11704-025-50297-0
  venue: Frontiers of Computer Science 21(4)
  first_date: '2024-09-05'
  access: arxiv
  origin: scoping
  closest: 0
- id: liao_2505.00976
  kind: survey
  title: 'Attack and defense techniques in large language models: A survey and new perspectives'
  first_author: Liao
  arxiv: '2505.00976'
  doi: 10.1016/j.neunet.2025.108388
  venue: Neural Networks
  first_date: '2025-05-02'
  access: arxiv
  origin: scoping
  closest: 0
- id: aguilera_2505.01177
  kind: survey
  title: 'LLM Security: Vulnerabilities, Attacks, Defenses, and Countermeasures'
  first_author: Aguilera-Martínez
  arxiv: '2505.01177'
  doi: ''
  venue: arXiv
  first_date: '2025-05-02'
  access: arxiv
  origin: scoping
  closest: 0
- id: li_fung_2505.18889
  kind: survey
  title: 'Security Concerns for Large Language Models: A Survey'
  first_author: Li
  arxiv: '2505.18889'
  doi: ''
  venue: arXiv
  first_date: 2025-05
  access: arxiv
  origin: scoping
  closest: 0
- id: nasr_2510.09023
  kind: evaluation
  title: 'The Attacker Moves Second: Stronger Adaptive Attacks Bypass Defenses Against LLM Jailbreaks and Prompt Injections'
  first_author: Nasr
  arxiv: '2510.09023'
  doi: ''
  venue: arXiv (cited as USENIX Security 2026; not verified)
  first_date: '2025-10-10'
  access: arxiv
  origin: scoping
  closest: 0
- id: li_s_2601.07185
  kind: evaluation
  title: Defenses Against Prompt Attacks Learn Surface Heuristics
  first_author: Li
  arxiv: '2601.07185'
  doi: ''
  venue: arXiv
  first_date: '2026-01-12'
  access: arxiv
  origin: scoping
  closest: 0
- id: attackeval_2604.03598
  kind: evaluation
  title: 'AttackEval: A Systematic Empirical Study of Prompt Injection Attack Effectiveness'
  first_author: ''
  arxiv: '2604.03598'
  doi: ''
  venue: arXiv
  first_date: 2026-04
  access: arxiv
  origin: scoping
  closest: 0
- id: correia_2601.22240
  kind: survey
  title: 'A Systematic Literature Review on LLM Defenses Against Prompt Injection and Jailbreaking: Expanding NIST Taxonomy'
  first_author: Correia
  arxiv: '2601.22240'
  doi: ''
  venue: arXiv (states submitted to Computer Science Review)
  first_date: '2026-01-29'
  access: arxiv
  origin: second
  closest: 1
- id: hashemi_2606.31639
  kind: survey
  title: 'A Lifecycle and Application-Stack Survey of Large Language Model Vulnerabilities: Attacks, Risks, Defenses, and
    Open Problems'
  first_author: Hashemi Natanzi
  arxiv: '2606.31639'
  doi: ''
  venue: arXiv
  first_date: '2026-06-30'
  access: arxiv
  origin: second
  closest: 1
- id: gulyamov_information_2026
  kind: survey
  title: 'Prompt Injection Attacks in Large Language Models and AI Agent Systems: A Comprehensive Review of Vulnerabilities,
    Attack Vectors, and Defense Mechanisms'
  first_author: Gulyamov
  arxiv: ''
  doi: 10.3390/info17010054
  venue: Information 17(1):54
  first_date: '2026-01-07'
  access: open
  pdf_url: https://www.mdpi.com/2078-2489/17/1/54/pdf
  origin: second
  closest: 0
- id: woesle_ieeeaccess_2026
  kind: survey
  title: A Systematic Literature Review of Prompt Injection Attacks in LLM-Integrated Systems
  first_author: Woesle
  arxiv: ''
  doi: 10.1109/ACCESS.2026.3700492
  venue: IEEE Access 14:86797-86818
  first_date: '2026-06-04'
  access: open
  pdf_url: ''
  origin: second
  closest: 0
- id: duarte_ieeeaccess_2026
  kind: survey
  title: 'A Systematic Review of Prompt Injection Attacks on Large Language Models: Trends, Taxonomy, Evaluation, Defenses
    and Opportunities'
  first_author: Duarte
  arxiv: ''
  doi: 10.1109/ACCESS.2026.3656849
  venue: IEEE Access 14:12875-12899
  first_date: '2026'
  access: open
  pdf_url: ''
  origin: second
  closest: 0
- id: mourad_electronics_2026
  kind: survey
  title: 'Securing the Prompt Pipeline: A Systematic Review of Defense Mechanisms Against Prompt-Based Attacks in LLM Agents'
  first_author: Mourad
  arxiv: ''
  doi: 10.3390/electronics15163700
  venue: Electronics 15(16):3700
  first_date: '2026-08-19'
  access: open
  pdf_url: https://www.mdpi.com/2079-9292/15/16/3700/pdf
  origin: second
  closest: 1
- id: zenodo_19856514
  kind: survey
  title: 'Prompt injection attacks and defenses in large language models: A systematic literature review'
  first_author: ''
  arxiv: ''
  doi: 10.5281/zenodo.19856514
  venue: Zenodo record 19856514
  first_date: ''
  access: unknown
  pdf_url: ''
  origin: second
  closest: 0
- id: supplychain_2602.19555
  kind: sok
  title: 'SoK: A Taxonomy of Attack Vectors and Defense Strategies for Agentic Supply Chain Runtime'
  first_author: ''
  arxiv: '2602.19555'
  doi: ''
  venue: arXiv
  first_date: 2026-02
  access: arxiv
  origin: second
  closest: 0
- id: trustworthy_2609.13731
  kind: survey
  title: 'Trustworthy Agentic AI: A Comprehensive Cybersecurity and Systems Survey on Threat Landscapes, Defense Architectures,
    and Open Challenges'
  first_author: ''
  arxiv: '2609.13731'
  doi: ''
  venue: arXiv
  first_date: 2026-09
  access: arxiv
  origin: second
  closest: 0
- id: dots_2609.23894
  kind: survey
  title: 'Connecting the Dots in Agentic AI Security: A Cross-Dimensional Threat Taxonomy, Evaluation Maturity, and Open Challenges'
  first_author: ''
  arxiv: '2609.23894'
  doi: ''
  venue: arXiv
  first_date: 2026-09
  access: arxiv
  origin: second
  closest: 0
- id: safety_at_scale_2502.05206
  kind: survey
  title: 'Safety at Scale: A Comprehensive Survey of Large Model and Agent Safety'
  first_author: ''
  arxiv: '2502.05206'
  doi: ''
  venue: arXiv
  first_date: 2025-02
  access: arxiv
  origin: second
  closest: 0
- id: ai_browsers_researchgate_2026
  kind: survey
  title: 'Securing LLM Powered AI Browsers Against Prompt Injection: A Comprehensive Survey, Threat Taxonomy, and Defense
    Framework'
  first_author: ''
  arxiv: ''
  doi: ''
  venue: ResearchGate only; no publisher page found
  first_date: 2026-01
  access: unknown
  pdf_url: ''
  origin: second
  closest: 0
- id: ling_2606.10749
  kind: survey
  title: 'Toward secure LLM agents: Threat surfaces, attacks, defenses, and evaluation'
  first_author: Ling
  arxiv: '2606.10749'
  doi: ''
  venue: arXiv
  first_date: 2026-06
  access: arxiv
  origin: second
  closest: 0
- id: milani_ncaa_2026
  kind: not_survey
  title: Indirect prompt injection in large language models
  first_author: Milani
  arxiv: ''
  doi: 10.1007/s00521-026-12266-x
  venue: Neural Computing and Applications 38:530
  first_date: '2026'
  access: unknown
  pdf_url: ''
  origin: second
  closest: 0
- id: jia_2505.18333
  kind: evaluation
  title: A Critical Evaluation of Defenses against Prompt Injection Attacks
  first_author: Jia
  arxiv: '2505.18333'
  doi: ''
  venue: arXiv
  first_date: '2025-05-23'
  access: arxiv
  origin: second
  closest: 0
- id: yang_2505.15738
  kind: evaluation
  title: 'Auditing Prompt Injection Defenses Before They Reach Users: What Strong Adversaries Reveal'
  first_author: Yang
  arxiv: '2505.15738'
  doi: ''
  venue: arXiv
  first_date: '2025-05-21'
  access: arxiv
  origin: second
  closest: 0
- id: piarena_2604.08499
  kind: evaluation
  title: 'PIArena: A Platform for Prompt Injection Evaluation'
  first_author: ''
  arxiv: '2604.08499'
  doi: ''
  venue: arXiv
  first_date: 2026-04
  access: arxiv
  origin: second
  closest: 0
- id: oob_2606.26479
  kind: evaluation
  title: Adaptive Evaluation of Out-of-Band Defenses Against Prompt Injection in LLM Agents
  first_author: ''
  arxiv: '2606.26479'
  doi: ''
  venue: arXiv
  first_date: 2026-06
  access: arxiv
  origin: second
  closest: 0
- id: autodojo_2606.15057
  kind: evaluation
  title: AutoDojo
  first_author: Ma
  arxiv: '2606.15057'
  doi: ''
  venue: arXiv
  first_date: 2026-06
  access: arxiv
  origin: second
  closest: 0
- id: pismith_2603.13026
  kind: evaluation
  title: 'PISmith: Reinforcement Learning-based Red Teaming for Prompt Injection Defenses'
  first_author: ''
  arxiv: '2603.13026'
  doi: ''
  venue: arXiv
  first_date: 2026-03
  access: arxiv
  origin: second
  closest: 0
- id: attention_attack_2507.07417
  kind: evaluation
  title: May I have your Attention? Breaking Fine-Tuning based Prompt Injection Defenses using Architecture-Aware Attacks
  first_author: ''
  arxiv: '2507.07417'
  doi: ''
  venue: arXiv
  first_date: 2025-07
  access: arxiv
  origin: second
  closest: 0
- id: secfid_2606.30783
  kind: benchmark
  title: 'Security-Fidelity Tradeoffs: The Hidden Cost of Prompt Injection Defense'
  first_author: Hermon
  arxiv: '2606.30783'
  doi: ''
  venue: ICML 2026 (PMLR 306)
  first_date: 2026-06
  access: arxiv
  origin: second
  closest: 1

In [ ]:
%%writefile schema/overlap_prefill.yaml
wang_2602.10453:
  scope: Prompt-injection threats and defences in LLM agents; 78 papers; defences by the stage at which they act.
  corpus_cutoff: early 2026 (to confirm from the PDF)
  organising_dimensions: 'Attacks by payload generation. Defences by stage: text level (detection filters, prompt
    enhancement), model level (model alignment by task, preference or format; internal-representation detectors),
    execution level (task alignment, access control, isolation).'
  treatment_user_task: Utility without attack as a general score on AgentPI; not the user's instruction under attack
    as its own quantity.
  treatment_document_handling: Not measured. The paper finds that defences look good by suppressing contextual inputs,
    which is the symptom.
  treatment_comparability: Prompt-structure changes counted as a compatibility cost; layout not treated as a condition
    for comparing numbers.
  evidence_granularity: Benchmark results (ASR, utility, time, tokens) for eight text- and execution-level defences;
    no model-level defence run.
  what_this_review_adds: Finer first axis with no-signal as a value; signal dependence; R2 and R3 coverage; evidence
    grades for comparisons.
  a_source_axis: partial
  b_three_jobs_together: 'no'
  c_equal_layout_rule: 'no'
  read_basis: full PDF (research pass, 19 Nov 2026)
ji_2511.15203:
  scope: Defence frameworks for indirect prompt injection in LLM agents.
  corpus_cutoff: late 2025 (to confirm)
  organising_dimensions: 'Five dimensions: technical paradigm, intervention stage, model access, explainability,
    automation level.'
  treatment_user_task: Utility on AgentDojo (task success) for nine frameworks; not the user's instruction as its
    own quantity.
  treatment_document_handling: Not measured.
  treatment_comparability: Prompt templates and delimiters appear as implementation details; no comparability condition.
  evidence_granularity: ASR and utility for nine frameworks under three adaptive attacks; Meta-SecAlign the only
    training defence.
  what_this_review_adds: Model-side defences as the subject; the two axes; R2 and R3; grades.
  a_source_axis: partial
  b_three_jobs_together: 'no'
  c_equal_layout_rule: 'no'
  read_basis: full PDF (research pass, 19 Nov 2026)
correia_2601.22240:
  scope: Systematic literature review of LLM defences against prompt injection and jailbreaking; 88 studies to August
    2025.
  corpus_cutoff: August 2025
  organising_dimensions: 'NIST-based tree: training-time interventions; deployment-time interventions (prompt instruction
    and formatting, input modification, aggregation, model-level mitigations such as decoding steering and defensive
    pruning, detection, monitoring); indirect mitigations.'
  treatment_user_task: ASR is the main metric; some studies also report performance; trade-offs often not reported
    (p. 15).
  treatment_document_handling: Not treated.
  treatment_comparability: RQ3 asks whether results are comparable; Section 6.2 names dataset splits, model variants
    and temperature; prompt layout is not among the factors.
  evidence_granularity: Catalogue with model, attack dataset, code and model-agnosticism per study (Table 3).
  what_this_review_adds: The inference-time signal axis separated from the training axis; R2 and R3; layout as a
    condition; grades.
  a_source_axis: partial
  b_three_jobs_together: 'no'
  c_equal_layout_rule: 'no'
  read_basis: full PDF (research pass, 19 Nov 2026)
hashemi_2606.31639:
  scope: Lifecycle and application-stack survey of LLM vulnerabilities; systematisation rather than exhaustive bibliometrics.
  corpus_cutoff: mid 2026 (to confirm)
  organising_dimensions: 'Defence-in-depth: deterministic controls; model-level and prompt-level controls; retrieval
    and memory controls; tool and agent controls; monitoring, red teaming, governance.'
  treatment_user_task: No numbers reported.
  treatment_document_handling: Not treated.
  treatment_comparability: A reporting checklist (Table 5) lists context construction including delimiters and templates
    as items to report; not a comparability condition.
  evidence_granularity: Narrative; no extracted evidence tables.
  what_this_review_adds: Measured coverage of the three jobs; graded comparisons; the card as a filled artefact.
  a_source_axis: 'no'
  b_three_jobs_together: 'no'
  c_equal_layout_rule: partial
  read_basis: full PDF (research pass, 19 Nov 2026)
mourad_electronics_2026:
  scope: Systematic review of defence mechanisms against prompt-based attacks in LLM agents; 19 studies.
  corpus_cutoff: to confirm from the PDF
  organising_dimensions: Pipeline stage (prompt, model, pipeline; Section 1.1); system architecture (single-LLM,
    multi-agent); technical approach (ML-based, rule-based, hybrid).
  treatment_user_task: unknown (PDF not yet read)
  treatment_document_handling: unknown (PDF not yet read)
  treatment_comparability: unknown (PDF not yet read)
  evidence_granularity: unknown (PDF not yet read)
  what_this_review_adds: to fill after reading
  a_source_axis: partial
  b_three_jobs_together: unknown
  c_equal_layout_rule: unknown
  read_basis: MDPI landing page (19 Nov 2026)
geng_cmc_2026:
  scope: Survey of prompt injection attack methods, root causes and defences; 128 studies 2022 to 2025.
  corpus_cutoff: '2025'
  organising_dimensions: Attacks by vector, objective and implementation; defences in three groups (input filtering,
    system architecture, model level); root causes at philosophical, architectural and training levels.
  treatment_user_task: unknown (PDF not yet read)
  treatment_document_handling: unknown (PDF not yet read)
  treatment_comparability: unknown (PDF not yet read)
  evidence_granularity: unknown (PDF not yet read)
  what_this_review_adds: to fill after reading
  a_source_axis: partial
  b_three_jobs_together: unknown
  c_equal_layout_rule: unknown
  read_basis: journal page (19 Nov 2026)
sammartino_kbs_2026:
  scope: PRISMA-informed review of threats and defences in LLM-driven agentic and multi-agent systems, 2023 to early
    2026.
  corpus_cutoff: early 2026
  organising_dimensions: Threats by entry point, propagation path and target asset; four threat families; five defence
    categories mapped to threats.
  treatment_user_task: unknown (full text paywalled)
  treatment_document_handling: unknown (full text paywalled)
  treatment_comparability: Defences assessed for adaptive-attack resistance, cost and residual risk (abstract);
    layout not mentioned in the abstract.
  evidence_granularity: unknown (full text paywalled)
  what_this_review_adds: to fill after reading
  a_source_axis: unknown
  b_three_jobs_together: unknown
  c_equal_layout_rule: unknown
  read_basis: journal abstract page (19 Nov 2026)
hong_2510.15476:
  scope: SoK of LLM prompt security, jailbreak-centred; 272 studies; three taxonomies; unified evaluation.
  corpus_cutoff: 2025 (v4 October 2026; to confirm)
  organising_dimensions: Taxonomies of attacks, defences (by goal and methodology; a defence decision layer of input,
    output, internal state or prompt policy) and vulnerabilities; an evaluation protocol over model, access setting,
    attack, defence, sample and judge.
  treatment_user_task: Not as its own quantity (to confirm).
  treatment_document_handling: Not treated (jailbreak setting).
  treatment_comparability: Configuration-aware reporting over the tuple; prompt layout is not in the tuple.
  evidence_granularity: 11 models, 20 attacks, 9 defences, 3 judges under one protocol.
  what_this_review_adds: Indirect injection as the subject; the signal axis; R2 and R3; layout as a condition.
  a_source_axis: partial
  b_three_jobs_together: 'no'
  c_equal_layout_rule: partial
  read_basis: arXiv abstract page and v3 HTML snippets (19 Nov 2026)
secfid_2606.30783:
  scope: 'Benchmark, not a survey: security and fidelity measured together for four defences on three bases; 1,168
    instances.'
  corpus_cutoff: not applicable
  organising_dimensions: Outcomes executed, processed, ignored, other; fidelity as one minus the ignored rate (non-suppression);
    safe processing as processed and not executed.
  treatment_user_task: Not measured separately.
  treatment_document_handling: 'Measured: non-suppression and safe processing on translation, editing, extraction
    and counting.'
  treatment_comparability: Each defence run with its own specified inference procedure (Appendix B.1).
  evidence_granularity: 'Per defence and base model: security and fidelity percentages.'
  what_this_review_adds: SecFid is the source of the R3 definitions and of published pairs; the review places its
    numbers on the map with their layouts.
  a_source_axis: 'no'
  b_three_jobs_together: partial
  c_equal_layout_rule: 'no'
  read_basis: full PDF (research pass, 19 Nov 2026)

## Stage 2. Candidate records with versions linked and arXiv metadata

In [ ]:
import yaml, json
import corpus, overlap, paths
importlib.reload(corpus); importlib.reload(overlap); importlib.reload(paths)
paths.ensure_dirs()

seed = yaml.safe_load(Path("schema/candidates_seed.yaml").read_text())
rows = [dict(r) for r in seed]
rows = corpus.link_versions(rows)

arxiv_ids = [r["arxiv"] for r in rows if r.get("arxiv")]
meta = corpus.fetch_arxiv_metadata(arxiv_ids, paths.REPORTS / "arxiv_meta.json")
for r in rows:
    m = meta.get(r.get("arxiv", ""), {})
    r["arxiv_title"] = m.get("title", "")
    r["arxiv_published"] = m.get("published", "")
    r["arxiv_updated"] = m.get("updated", "")
    r["arxiv_latest_version"] = m.get("latest_version", "")
    r["arxiv_journal_ref"] = m.get("journal_ref", "")
    r["arxiv_doi"] = m.get("doi", "")
corpus.write_csv(paths.REPORTS / "candidates.csv", rows, corpus.CANDIDATE_FIELDS)

kinds = {}
for r in rows:
    kinds[r["kind"]] = kinds.get(r["kind"], 0) + 1
print("candidate records:", len(rows), kinds)
print("arXiv metadata found for", sum(1 for r in rows if r["arxiv_title"]), "of", len(arxiv_ids), "arXiv ids",
      "(offline)" if corpus.OFFLINE else "")
print("linked versions:", [(r["id"], r["linked_to"]) for r in rows if r["linked_to"]] or "none")
title_mismatch = [(r["id"], r["arxiv_title"]) for r in rows
                  if r["arxiv_title"] and corpus.normalise_title(r["arxiv_title"]) != corpus.normalise_title(r["title"])]
print("seed titles that differ from the arXiv title:", len(title_mismatch))
for item in title_mismatch:
    print("  ", item[0], "->", item[1][:100])

## Stage 3. PDFs, hashes and page counts

In [ ]:
survey_rows = []
status_counts = {}
for r in rows:
    dest = paths.DATA_SURVEYS / f"{r['id']}.pdf"
    url = corpus.pdf_url_for(r)
    if url:
        status = corpus.download_pdf(url, dest)
    else:
        status = "present (hand-placed)" if dest.exists() else "missing: place the PDF by hand"
    r = dict(r)
    r["pdf_path"] = str(dest.relative_to(paths.REPO_ROOT)) if dest.exists() else ""
    r["pdf_status"] = status
    r["sha256"] = corpus.sha256(dest) if dest.exists() else ""
    r["pages"] = corpus.page_count(dest) if dest.exists() else ""
    survey_rows.append(r)
    key = status.split(":")[0]
    status_counts[key] = status_counts.get(key, 0) + 1
corpus.write_csv(paths.REPORTS / "surveys.csv", survey_rows, corpus.SURVEY_FIELDS)
print("PDF status:", status_counts)
for r in survey_rows:
    if r["pdf_status"].startswith("failed") or r["pdf_status"].startswith("missing"):
        print("  ", r["id"], "->", r["pdf_status"][:120])
closest_missing = [r["id"] for r in survey_rows if str(r.get("closest")) == "1" and not r["sha256"]]
print("closest items without a PDF on disk:", closest_missing or "none")

## Stage 4. Overlap forms for the closest items

In [ ]:
prefill = yaml.safe_load(Path("schema/overlap_prefill.yaml").read_text())
by_id = {r["id"]: r for r in survey_rows}
template = overlap.blank_form("<record id>", "<title>", "<kind>")
Path("schema/overlap_form_template.yaml").write_text(yaml.safe_dump(template, sort_keys=False, allow_unicode=True))

created = []
for record_id, values in prefill.items():
    r = by_id[record_id]
    form = overlap.blank_form(record_id, r["title"], r["kind"])
    form.update(values)
    form["pdf_sha256"] = ""          # filled when the form is confirmed from the PDF on disk
    form["confirmed_from_pdf"] = False
    if overlap.write_form_if_missing(paths.REPORTS / "overlap" / f"{record_id}.yaml", form):
        created.append(record_id)
closest_without_form = [r["id"] for r in survey_rows if str(r.get("closest")) == "1" and r["id"] not in prefill]
print("forms created now:", created or "none (all present)")
print("closest items without a prefilled form:", closest_without_form or "none")
print("forms on disk:", len(list((paths.REPORTS / "overlap").glob("*.yaml"))))

## Stage 5. Gate G1

In [ ]:
import datetime as dt
forms = overlap.load_forms(paths.REPORTS / "overlap")
outcome, reasons = overlap.decide(forms)
when = dt.date.today().isoformat()
(paths.REPORTS / "overlap_01.md").write_text(overlap.render_markdown(forms, outcome, reasons, when))
lines = [f"decision_01: {outcome}", f"date: {when}", f"forms: {len(forms)}"] + [f"- {r}" for r in reasons]
(paths.REPORTS / "decision_01.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
problems = {f["id"]: overlap.validate(f) for f in forms}
problems = {k: v for k, v in problems.items() if v}
print("forms with validation notes:", len(problems))
for k, v in problems.items():
    print("  ", k, "->", "; ".join(v)[:160])
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 6. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "01 candidates and overlap: seed records, arXiv metadata, PDF manifest, prefilled overlap forms, gate G1"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)